# Ajaliselt korrektne tunnuste konstrueerimine

Prognoos tehakse hetkel t järgmise tunni sihile t+1. Baas on fikseeritud; kandidaatide rühmi lisatakse hiljem kontrollitud katsetes. Järgmise tunni mõõdetud ilm ja tegelik koormusmuutus on ainult tagasivaatelised sildid.

In [ ]:
from pathlib import Path
from functools import partial
import sys
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_run

RUN = latest_run()
table = partial(show_table, RUN, limit=30)
show_run(RUN)

## Tunnuste rühmad ja info kättesaadavus

Ilma viivitus on 24 tundi. Koormus t on eeldatavalt kohe teada. Püha ja kooli puhul kasutatakse sama päeva teadaolevat märgendit; kuupäevapiiril on asendus 0. Avaldamisaja eeldused piiravad operatiivset rakendatavust.

In [ ]:
table(
    "processed/feature_manifest.csv",
    rename={"timing": "Ajaline kättesaadavus", "limitation": "Piirang"},
    limit=50,
)
display(
    Markdown(
        "Tabel eristab 13 baastunnust ja kuut kandidaatrühma. Tunnuse ajaline lubatavus ei taga ennustusväärtust. Sama kandidaatide hulk jääb kättesaadavaks kõigile kolmele valikumeetodile, et RQ2 võrdlus oleks kontrollitud."
    )
)

## Nihked ja ühine vaatlusmask

Eelmise päeva sihttund on t−23 ja eelmise nädala sihttund t−167. Rolling-aken lõpeb t juures; tulevikuandmed ei kuulu aknasse. Kõik mudelid kasutavad sama vaatlusmaski, ka siis, kui lihtsam mudel vajaks vähem ajalugu.

In [ ]:
accounting = read_json(RUN / "processed/row_accounting.json")
display(
    Markdown(
        f"Lähteridu {accounting['raw_rows']}, kasutatavaid prognoosiridu {accounting['usable_rows']}, ajaloo/sihtide servadest välja jäänud ridu {accounting['excluded_boundary_rows']}. Põhjus: {accounting['reason']} Ühtegi sisemist anomaaliat selle maskiga ei eemaldata."
    )
)
from src.feature_engineering import load_run_features, split_features

config = read_json(RUN / "configs/experiment.json")
train = split_features(load_run_features(RUN), config)["train"]
columns = ["origin_time", "target_time", "load_now", "load_day", "load_week", "target_next_hour"]
display(
    train[columns]
    .head(5)
    .rename(
        columns={
            "origin_time": "Prognoosihetk",
            "target_time": "Sihtaeg",
            "load_now": "Viimane koormus",
            "load_day": "Päeva sihttunni minevik",
            "load_week": "Nädala sihttunni minevik",
            "target_next_hour": "Tegelik koormus",
        }
    )
)
display(
    Markdown(
        "Tabel võimaldab kontrollida sihi ja minevikukoormuste nihkeid. Need on arvutuslikud kontrollread, mitte mudeli kvaliteeditõend. Ühiktest muudab kõiki tulevasi väärtusi ning kontrollib, et varasem sisendrida ei muutu."
    )
)

## Tunnuste lisakatsete asukoht

Põhiosas võrdleme algseid tunnuseid koormusajaloo ja kalendriga. Tunnuserühmade üksikkatsed ja eemalduskatsed on [lisas 09](09_expanded_feature_study.ipynb). Samu tabeleid ja jooniseid siin ei korrata.